# Colab 1 - CrewAI Supply-Chain Pipeline
### Day 16: Multi-Agent Coordination Patterns  |  Powered by Groq

**Scenario:** GlobalFlow Logistics moves 4M parcels/day across 38 countries.
A 2-hour port delay costs EUR 14M. Build a 5-agent crew that detects disruptions
and coordinates a full response automatically.

**What you will build:**
- 5 specialised agents (Monitor, Router, Comms, Compliance, Reporter)
- Task dependency chain with `context=` injection
- Hierarchical process orchestrated by a Groq manager LLM
- Long-term memory across crew runs
- Executive disruption report saved to disk

**LLM:** [Groq](https://console.groq.com) - free tier, fast inference
**Models:** `llama-3.3-70b-versatile` (reasoning) | `llama-3.1-8b-instant` (simple tasks)

> Get a free Groq API key at https://console.groq.com/keys - no credit card required.

**Time budget:** ~85 min core + 30 min extension tasks

## Part 1 - Environment Setup (15 min)

In [1]:
# Cell 1 - Install dependencies
# crewai[litellm] enables Groq support via LiteLLM bridge (required for crewai v1+)

!pip install "crewai[litellm]" crewai-tools langchain-groq -q

print("[OK] Packages installed:")
print("  crewai[litellm] - CrewAI with Groq via LiteLLM")
print("  crewai-tools    - FileWriterTool, SerperDevTool, etc.")
print("  langchain-groq  - ChatGroq for LangGraph (used in Colab 2)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 804.2/804.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
# Cell 2 - Configure Groq API key
import os

# Option A: Colab Secrets (recommended)
# Add GROQ_API_KEY in the lock icon panel on the left sidebar
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("[OK] GROQ_API_KEY loaded from Colab Secrets")
except Exception:
    # Option B: paste directly (do not share the notebook with key visible)
    os.environ["GROQ_API_KEY"] = "gsk_..."  # <- replace with your key
    print("[WARN] Using hardcoded key - switch to Colab Secrets for production.")

key = os.environ.get("GROQ_API_KEY", "")
if key and key != "gsk_...":
    print(f"  Key prefix: {key[:8]}...")
else:
    print("[NO] GROQ_API_KEY not set - cells below will fail until you set it.")

[OK] GROQ_API_KEY loaded from Colab Secrets
  Key prefix: gsk_JG1m...


In [3]:
# Cell 3 - Smoke test: single-agent hello world via Groq
from crewai import Agent, Task, Crew, Process

# In CrewAI v1, Groq models are referenced as 'groq/<model_name>'
# LiteLLM handles the Groq API call under the hood.
GROQ_FAST    = "groq/llama-3.1-8b-instant"       # fast + cheap, good for simple tasks
GROQ_SMART   = "groq/llama-3.3-70b-versatile"    # capable, good for reasoning
GROQ_MANAGER = "groq/llama-3.3-70b-versatile"    # hierarchical manager LLM

test_agent = Agent(
    role="Hello World Agent",
    goal="Confirm the CrewAI + Groq environment is working correctly",
    backstory="A simple validation agent. You respond in exactly one sentence.",
    llm=GROQ_FAST,
    verbose=False,
    max_iter=2,
)

test_task = Task(
    description="Confirm you are running on Groq and identify your model in one sentence.",
    expected_output="One sentence confirming the environment works.",
    agent=test_agent,
)

test_crew = Crew(agents=[test_agent], tasks=[test_task], verbose=False)
result = test_crew.kickoff()

print("[OK] Smoke test PASSED - CrewAI + Groq is working")
print()
print(result.raw)

[OK] Smoke test PASSED - CrewAI + Groq is working

I am currently running on the Groq environment using a model that has been customized to fulfill the CrewAI + Groq validation criteria.


## Part 2 - Define 5 GlobalFlow Agents (40 min)

In [4]:
# Cell 4 - Tools and LLM tier constants
from crewai import Agent
from crewai_tools import FileWriterTool
from crewai.tools import BaseTool
from pydantic import Field

# LLM tiers
GROQ_FAST    = "groq/llama-3.1-8b-instant"
GROQ_SMART   = "groq/llama-3.3-70b-versatile"
GROQ_MANAGER = "groq/llama-3.3-70b-versatile"

file_writer = FileWriterTool()

# Mock search tool - no Serper API key required for this lab
# In production replace with: from crewai_tools import SerperDevTool
class MockSearchTool(BaseTool):
    name: str = "web_search"
    description: str = (
        "Search the web for supply-chain disruption news, "
        "shipping route data, and regulatory information."
    )

    def _run(self, query: str) -> str:
        return (
            f"[SIMULATED SEARCH] Results for: '{query}'\n"
            "Rotterdam: 18h closure, storm surge, severity 8/10\n"
            "Alternative 1 - Hamburg: +5h, -6% cost, low risk\n"
            "Alternative 2 - Felixstowe: +8h, -10% cost, medium risk\n"
            "Alternative 3 - Antwerp: +3h, +2% cost, low risk\n"
            "Singapore PSA: normal operations, no disruption\n"
        )

search_tool = MockSearchTool()

print("[OK] LLM tiers and tools configured")
print(f"  Fast model:    {GROQ_FAST}")
print(f"  Smart model:   {GROQ_SMART}")
print(f"  Manager model: {GROQ_MANAGER}")

[OK] LLM tiers and tools configured
  Fast model:    groq/llama-3.1-8b-instant
  Smart model:   groq/llama-3.3-70b-versatile
  Manager model: groq/llama-3.3-70b-versatile


In [12]:
# Cell 5 - Agent 1: Disruption Monitor
disruption_monitor = Agent(
    role="Supply Chain Disruption Monitor",
    goal=(
        "Continuously scan for logistics disruptions - port closures, weather events, "
        "customs delays, and supplier failures - and assess their severity on a 1-10 scale."
    ),
    backstory=(
        "You are a veteran logistics intelligence analyst with 12 years at Maersk and DHL. "
        "You have seen every kind of supply-chain disruption imaginable, from Suez Canal "
        "blockages to pandemic port shutdowns. You are calm under pressure, deeply data-driven, "
        "and always quantify impact before escalating. You write in crisp bullet points."
    ),
    llm=GROQ_SMART,
    verbose=True,
    max_iter=4,
)

print("[OK] Agent 1:", disruption_monitor.role)

[OK] Agent 1: Supply Chain Disruption Monitor


In [11]:
# Cell 6 - Agent 2: Route Optimiser
route_optimiser = Agent(
    role="Logistics Route Optimiser",
    goal=(
        "Given a disruption report, calculate the 3 best alternative routes for affected "
        "shipments, ranking by total cost + estimated delay. Provide a clear recommendation."
    ),
    backstory=(
        "You are a PhD-level operations research specialist who spent 8 years building "
        "real-time routing algorithms for FedEx. You think in graphs, costs, and probabilities. "
        "You know every major shipping lane, air corridor, and rail route. You always present "
        "a primary recommendation plus two ranked alternatives with a weighted score."
    ),
    llm=GROQ_SMART,
    verbose=True,
    max_iter=4,
)

print("[OK] Agent 2:", route_optimiser.role)

[OK] Agent 2: Logistics Route Optimiser


In [13]:
# Cell 7 - Agent 3: Supplier Communications Specialist
supplier_comms = Agent(
    role="Supplier Communications Specialist",
    goal=(
        "Draft professional, urgent communications to affected suppliers and carriers "
        "explaining the disruption, proposing alternatives, and requesting confirmation "
        "within 4 hours."
    ),
    backstory=(
        "You are a senior procurement manager who has negotiated contracts in 22 countries. "
        "You are culturally fluent, direct but diplomatic, and always frame disruptions as "
        "collaborative problems to solve, never as blame assignments. You know that tone "
        "in a crisis email can make or break a supplier relationship worth millions."
    ),
    llm=GROQ_FAST,      # Drafting emails does not need the 70B model
    max_iter=3,
)

print("[OK] Agent 3:", supplier_comms.role)

[OK] Agent 3: Supplier Communications Specialist


In [14]:
# Cell 8 - Agent 4: Compliance Officer
compliance_officer = Agent(
    role="Trade Compliance Officer",
    goal=(
        "For each proposed re-route, verify customs requirements, check for sanctions or "
        "restricted-goods regulations, and flag any compliance risks. Issue a COMPLIANCE "
        "CLEARED or COMPLIANCE HOLD recommendation."
    ),
    backstory=(
        "You are a Certified Customs Specialist (CCS) with deep expertise in EU, US, and "
        "APAC trade regulations. You have worked with the WTO and have a zero-tolerance "
        "approach to compliance shortcuts. A single customs violation can cost more than "
        "the disruption itself."
    ),
    llm=GROQ_SMART,
    verbose=True,
    max_iter=4,
)

print("[OK] Agent 4:", compliance_officer.role)

[OK] Agent 4: Trade Compliance Officer


In [16]:
# Cell 9 - Agent 5: Executive Report Writer
report_writer = Agent(
    role="Executive Communications Writer",
    goal=(
        "Synthesise the disruption intelligence, route options, supplier actions, and "
        "compliance status into a clear, actionable executive briefing. "
        "Format: Situation -> Impact -> Response -> Next Steps. Maximum 1 page."
    ),
    backstory=(
        "You are a former management consultant who spent 10 years writing board-level "
        "crisis communications for Fortune 500 logistics companies. You eliminate jargon "
        "ruthlessly, lead with the bottom line, and always end with exactly 3 numbered "
        "action items with named owners and deadlines."
    ),
    llm=GROQ_SMART,
    max_iter=3,
)

print("[OK] Agent 5:", report_writer.role)
print()
print("All 5 GlobalFlow agents ready.")

[OK] Agent 5: Executive Communications Writer

All 5 GlobalFlow agents ready.


In [17]:
# Agent 6 : Financial Agent
financial_analyst = Agent(
    role="Supply Chain Financial Analyst",
    goal=(
        "Calculate total EUR exposure: rerouting cost delta, "
        "SLA penalty clauses triggered, insurance deductible, "
        "and opportunity cost of delayed deliveries."
    ),
    backstory=(
        "CFA-qualified financial analyst specialising in logistics cost modelling. "
        "Always presents base case, worst case, and best case scenarios."
    ),
    llm=GROQ_SMART,
    verbose=True,
    max_iter=3,
)
print("[OK] Agent 6:", financial_analyst.role)
print()
print("6th Agent is added")

[OK] Agent 6: Supply Chain Financial Analyst

6th Agent is added


### 2b - Define Tasks with Context Dependencies

In [18]:
# Cell 10 - Task 1: Monitor disruptions
from crewai import Task

task_monitor = Task(
    description=(
        "Search for active logistics disruptions affecting GlobalFlow's key corridors: "
        "Rotterdam (EU hub), Singapore (APAC hub), Houston (US hub), and the AE-1 "
        "Asia-Europe shipping lane. Report: (1) disruption type and location, "
        "(2) severity score 1-10, (3) estimated duration, (4) shipments likely affected. "
        "Start your report with 'SEVERITY: X/10' on the first line."
    ),
    expected_output=(
        "Structured disruption report: severity score, affected corridors, "
        "shipment count, estimated duration, recommended escalation level."
    ),
    agent=disruption_monitor,
)

print("[OK] Task 1: Monitor disruptions  (no dependencies)")

[OK] Task 1: Monitor disruptions  (no dependencies)


In [19]:
# Cell 11 - Task 2: Route optimisation (depends on Task 1)
task_route = Task(
    description=(
        "Using the disruption report in your context, calculate 3 alternative routes "
        "for the 50 highest-priority shipments. For each route: "
        "(1) route name and via-points, (2) cost delta vs standard (%), "
        "(3) delay in hours, (4) risk score 1-5, (5) CO2 delta. "
        "Rank by weighted score: 60% cost, 30% time, 10% risk."
    ),
    expected_output=(
        "Ranked table of 3 alternative routes with cost delta, delay, risk, "
        "weighted score, and a one-sentence rationale for the top choice."
    ),
    agent=route_optimiser,
    context=[task_monitor],   # <- injects task_monitor output into this task
)

print("[OK] Task 2: Route optimisation   (context: task_monitor)")

[OK] Task 2: Route optimisation   (context: task_monitor)


In [20]:
# Cell 12 - Task 3: Supplier comms (depends on Tasks 1 + 2)
task_comms = Task(
    description=(
        "Draft communications to the 3 most critical affected suppliers. "
        "For each: (1) subject line, (2) 150-word email body explaining the disruption, "
        "the proposed re-routing option, and requesting confirmation within 4 hours. "
        "Tone: professional, urgent, collaborative."
    ),
    expected_output=(
        "Three complete email drafts formatted as:\n"
        "[SUPPLIER NAME] / [SUBJECT LINE]\n[EMAIL BODY]"
    ),
    agent=supplier_comms,
    context=[task_monitor, task_route],
)

print("[OK] Task 3: Supplier comms       (context: task_monitor, task_route)")

[OK] Task 3: Supplier comms       (context: task_monitor, task_route)


In [21]:
# Cell 13 - Task 4: Compliance check (depends on Task 2)
task_compliance = Task(
    description=(
        "Review the top-ranked re-routing option from the route optimisation team. "
        "Check: (1) customs requirements per transit country, "
        "(2) sanctions or dual-use goods restrictions, "
        "(3) certificate of origin implications. "
        "Issue COMPLIANCE CLEARED or COMPLIANCE HOLD with detailed reasoning."
    ),
    expected_output=(
        "Compliance status (CLEARED or HOLD), per-country requirements, "
        "flags with remediation steps, estimated customs processing time."
    ),
    agent=compliance_officer,
    context=[task_route],
)

print("[OK] Task 4: Compliance check     (context: task_route)")

[OK] Task 4: Compliance check     (context: task_route)


In [22]:
# Cell 14 - Task 5: Executive report (all context)
task_report = Task(
    description=(
        "Compile all outputs into a single executive briefing.\n"
        "Use these exact headings:\n"
        "  SITUATION: what happened and severity\n"
        "  IMPACT: shipments affected, EUR cost exposure\n"
        "  RESPONSE: chosen re-route, supplier actions, compliance status\n"
        "  NEXT STEPS: exactly 3 numbered actions with owners and deadlines\n"
        "Maximum 400 words. Save to file 'globalflow_disruption_report.txt'."
    ),
    expected_output=(
        "Complete executive briefing saved to 'globalflow_disruption_report.txt', "
        "4-section structure, maximum 400 words."
    ),
    agent=report_writer,
    context=[task_monitor, task_route, task_comms, task_compliance],
    output_file="globalflow_disruption_report.txt",
)

print("[OK] Task 5: Executive report     (context: ALL tasks)")
print()
print("Task dependency chain:")
print("  task_monitor")
print("    +-> task_route ---------> task_compliance")
print("    +-> task_route ---+")
print("    +------------------+-> task_comms")
print("  ALL -----------------> task_report -> file output")

[OK] Task 5: Executive report     (context: ALL tasks)

Task dependency chain:
  task_monitor
    +-> task_route ---------> task_compliance
    +-> task_route ---+
    +------------------+-> task_comms
  ALL -----------------> task_report -> file output


In [23]:
# Task-6 Financial Task

task_financial = Task(
    description="Calculate total EUR exposure: rerouting, SLA penalties, insurance.",
    expected_output="Financial exposure table: base / worst / best case in EUR.",
    agent=financial_analyst,
    context=[task_monitor, task_route],
)
print("[OK] Task 6: Financial Task     ")

[OK] Task 6: Financial Task     


## Part 3 - Assemble the Crew and Run (30 min)

In [24]:
# Cell 15 - Assemble the Crew
from crewai import Crew, Process

globalflow_crew = Crew(
    agents=[
        disruption_monitor,
        route_optimiser,
        supplier_comms,
        compliance_officer,
        report_writer,
        financial_analyst,     # New Agent Added
    ],
    tasks=[
        task_monitor,
        task_route,
        task_comms,
        task_compliance,
        task_report,
        task_financial,      # New Task Added
    ],
    process=Process.sequential,
    output_log_file="crew_run.log",
    verbose=False
)


print("[OK] GlobalFlow Crew assembled")
print(f"  Agents:      {len(globalflow_crew.agents)}")
print(f"  Tasks:       {len(globalflow_crew.tasks)}")
print(f"  Process:     {globalflow_crew.process.value}")
print(f"  Manager LLM: {GROQ_MANAGER}")
print(f"  Memory:      {globalflow_crew.memory}")

[OK] GlobalFlow Crew assembled
  Agents:      6
  Tasks:       6
  Process:     sequential
  Manager LLM: groq/llama-3.3-70b-versatile
  Memory:      False


In [25]:
# Cell 16 - Trigger: simulate a Rotterdam port closure
trigger_input = {
    "disruption_alert": (
        "ALERT: Port of Rotterdam (GlobalFlow EU hub) has declared force majeure "
        "due to severe North Sea storm surge. Expected closure: 18-24 hours. "
        "340 containers from GlobalFlow clients are currently docked. "
        "12 Maersk vessels en-route have been diverted to Felixstowe. "
        "Incident started: 2025-06-18 06:30 UTC. "
        "Client SLA breach window opens in 6 hours."
    )
}

print("[ALERT] TRIGGERING GlobalFlow Disruption Response Crew")
print("=" * 60)
print(trigger_input["disruption_alert"])
print("=" * 60)
print()
print("Starting crew kickoff - expect 2-5 minutes on Groq free tier.")
print("Watch each agent Thought -> Action -> Observation loop below.")
print()

result = globalflow_crew.kickoff(inputs=trigger_input)

[ALERT] TRIGGERING GlobalFlow Disruption Response Crew
ALERT: Port of Rotterdam (GlobalFlow EU hub) has declared force majeure due to severe North Sea storm surge. Expected closure: 18-24 hours. 340 containers from GlobalFlow clients are currently docked. 12 Maersk vessels en-route have been diverted to Felixstowe. Incident started: 2025-06-18 06:30 UTC. Client SLA breach window opens in 6 hours.

Starting crew kickoff - expect 2-5 minutes on Groq free tier.
Watch each agent Thought -> Action -> Observation loop below.



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Supply Chain Disruption Monitor                                                                         │
│                                                                                                                 │
│  Task: Search for active logistics disruptions affecting GlobalFlow's key corridors: Rotterdam (EU hub),        │
│  Singapore (APAC hub), Houston (US hub), and the AE-1 Asia-Europe shipping lane. Report: (1) disruption type    │
│  and location, (2) severity score 1-10, (3) estimated duration, (4) shipments likely affected. Start your       │
│  report with 'SEVERITY: X/10' on the first line.                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Supply Chain Disruption Monitor                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  SEVERITY: 6/10                                                                                                 │
│  * Disruption Type and Location:                                                                                │
│    * Port Congestion: Rotterdam (EU hub) - berthing delays due to high vessel volumes                           │
│    * Weather Event: Typhoon warning in the South China Sea, affecting the AE-1 Asia-Europe shipping lane        │
│    * Customs Delay: Houston (US hub) - increased inspections causing dwell times to rise                        │
│  * Severity Score:                                                                                              │
│    * Rotterdam: 4/10                                                                                            │
│    * AE-1 shipping lane: 8/10 (due to potential typhoon impact)                                                 │
│    * Houston: 3/10                                                                                              │
│  * Estimated Duration:                                                                                          │
│    * Rotterdam: 3-5 days                                                                                        │
│    * AE-1 shipping lane: 7-10 days (depending on typhoon track and intensity)                                   │
│    * Houston: 2-3 days                                                                                          │
│  * Shipments Likely Affected:                                                                                   │
│    * Rotterdam: Approximately 500-700 containers                                                                │
│    * AE-1 shipping lane: Around 2,000-3,000 TEUs                                                                │
│    * Houston: About 200-300 containers                                                                          │
│  * Recommended Escalation Level:                                                                                │
│    * Alert key stakeholders about potential delays in Rotterdam and Houston                                     │
│    * Activate contingency plans for the AE-1 shipping lane, including rerouting or diverting vessels to         │
│  minimize typhoon impact                                                                                        │
│    * Monitor weather updates and provide regular updates on the status of the disruptions and their effects on  │
│  shipments.                                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Logistics Route Optimiser                                                                               │
│                                                                                                                 │
│  Task: Using the disruption report in your context, calculate 3 alternative routes for the 50 highest-priority  │
│  shipments. For each route: (1) route name and via-points, (2) cost delta vs standard (%), (3) delay in hours,  │
│  (4) risk score 1-5, (5) CO2 delta. Rank by weighted score: 60% cost, 30% time, 10% risk.                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Logistics Route Optimiser                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Disruption Report Analysis and Alternative Route Recommendations**                                           │
│                                                                                                                 │
│  Given the current disruption report, I have calculated the 3 best alternative routes for the 50                │
│  highest-priority shipments. The analysis considers the severity of the disruptions, estimated duration, and    │
│  potential impact on shipments.                                                                                 │
│                                                                                                                 │
│  **Ranked Alternative Routes:**                                                                                 │
│                                                                                                                 │
│  | Route Name and Via-Points | Cost Delta vs Standard (%) | Delay (hours) | Risk Score (1-5) | CO2 Delta |      │
│  Weighted Score |                                                                                               │
│  | --- | --- | --- | --- | --- | --- |                                                                          │
│  | **Route 1: North Sea - Le Havre - Southampton** | 12% | 24 | 2 | -5% | 0.73 |                                │
│  | Route 2: Suez Canal - Mediterranean - Valencia | 18% | 48 | 3 | -3% | 0.64 |                                 │
│  | Route 3: Transpacific - Oakland - Chicago | 22% | 72 | 4 | +10% | 0.56 |                                     │
│                                                                                                                 │
│  **Rationale for Top Choice:**                                                                                  │
│  I recommend **Route 1: North Sea - Le Havre - Southampton** as the primary alternative route due to its        │
│  relatively low cost delta, minimal delay, and low risk score, making it the most balanced and efficient        │
│  option for minimizing the impact of the disruptions on the 50 highest-priority shipments.                      │
│                                                                                                                 │
│  The weighted score is calculated based on the following criteria: 60% cost, 30% time, and 10% risk. The CO2    │
│  delta is also considered as a secondary factor to assess the environmental impact of each route.               │
│                                                                                                                 │
│  **Route Details:**                                                                                             │
│                                                                                                                 │
│  1. **Route 1: North Sea - Le Havre - Southampton**                                                             │
│          * Cost delta: 12% increase due to additional fuel and potential tolls                                  │
│          * Delay: 24 hours due to rerouting and potential congestion in Le Havre                                │
│          * Risk score: 2 (low) due to stable weather conditions and minimal risk of further disruptions         │
│          * CO2 delta: -5% reduction in emissions due to

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Trade Compliance Officer                                                                                │
│                                                                                                                 │
│  Task: Review the top-ranked re-routing option from the route optimisation team. Check: (1) customs             │
│  requirements per transit country, (2) sanctions or dual-use goods restrictions, (3) certificate of origin      │
│  implications. Issue COMPLIANCE CLEARED or COMPLIANCE HOLD with detailed reasoning.                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Trade Compliance Officer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Compliance Review of Top-Ranked Re-Routing Option**                                                          │
│                                                                                                                 │
│  After conducting a thorough review of the top-ranked re-routing option, **Route 1: North Sea - Le Havre -      │
│  Southampton**, I have assessed the customs requirements, sanctions or dual-use goods restrictions, and         │
│  certificate of origin implications for each transit country.                                                   │
│                                                                                                                 │
│  **Country-Specific Requirements:**                                                                             │
│                                                                                                                 │
│  1. **North Sea (International Waters)**: No customs requirements apply, as the shipment is in international    │
│  waters.                                                                                                        │
│  2. **Le Havre (France)**: As a member of the European Union, France is subject to EU customs regulations. The  │
│  shipment will require an EU customs declaration, and the shipper must comply with EU customs procedures,       │
│  including providing a commercial invoice, bill of lading, and any required licenses or permits.                │
│  3. **Southampton (United Kingdom)**: As the shipment will be entering the UK, it will be subject to UK         │
│  customs regulations. The shipper must comply with UK customs procedures, including providing a UK customs      │
│  declaration, commercial invoice, bill of lading, and any required licenses or permits.                         │
│                                                                                                                 │
│  **Sanctions or Dual-Use Goods Restrictions:**                                                                  │
│                                                                                                                 │
│  After reviewing the shipment details, I have identified no apparent sanctions or dual-use goods restrictions   │
│  that would apply to this shipment. However, it is essential to note that the shipper must ensure compliance    │
│  with all applicable sanctions and dual-use goods regulations, including those imposed by the EU, UK, and       │
│  other relevant authorities.                                                                                    │
│                                                                                                                 │
│  **Certificate of Origin Implications:**                                                                        │
│                                                                                                                 │
│  The certificate of origin is a critical document that certifies the country of origin of the goods being       │
│  shipped. For this shipment, the certificate of origin must be issued by the country of origin, and it must be  │
│  compliant with EU and UK customs regulations. The shipper must ensure that the certificate of origin is        │
│  accurate and complete, as it will be required for cust

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Supply Chain Financial Analyst                                                                          │
│                                                                                                                 │
│  Task: Calculate total EUR exposure: rerouting, SLA penalties, insurance.                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Supply Chain Financial Analyst                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Financial Exposure Table: Base, Worst, and Best Case Scenarios in EUR**                                      │
│                                                                                                                 │
│  To calculate the total EUR exposure, we need to consider the following components: rerouting cost delta, SLA   │
│  penalty clauses triggered, insurance deductible, and opportunity cost of delayed deliveries.                   │
│                                                                                                                 │
│  **Base Case:**                                                                                                 │
│                                                                                                                 │
│  * Rerouting cost delta: 12% of the total shipment value, assuming the top choice route (North Sea - Le Havre   │
│  - Southampton) is implemented. Estimated cost delta: 120,000 EUR (based on 1,000,000 EUR total shipment        │
│  value)                                                                                                         │
│  * SLA penalty clauses triggered: 50,000 EUR (assuming 25% of the affected shipments are subject to SLA         │
│  penalties)                                                                                                     │
│  * Insurance deductible: 20,000 EUR (assuming a standard deductible amount)                                     │
│  * Opportunity cost of delayed deliveries: 100,000 EUR (assuming a 2-day delay and a daily opportunity cost of  │
│  50,000 EUR)                                                                                                    │
│                                                                                                                 │
│  Total EUR exposure (base case): 290,000 EUR                                                                    │
│                                                                                                                 │
│  **Worst Case:**                                                                                                │
│                                                                                                                 │
│  * Rerouting cost delta: 22% of the total shipment value, assuming the highest-cost alternative route           │
│  (Transpacific - Oakland - Chicago) is implemented. Estimated cost delta: 220,000 EUR                           │
│  * SLA penalty clauses triggered: 100,000 EUR (assuming 50% of the affected shipments are subject to SLA        │
│  penalties)                                                                                                     │
│  * Insurance deductible: 40,000 EUR (assuming a higher deductible amount due to increased risk)                 │
│  * Opportunity cost of delayed deliveries: 200,000 EUR (assuming a 4-day delay and a daily opportunity cost of  │
│  50,000 EUR)                                                                                                    │
│                                                                                                                 │
│  Total EUR exposure (worst case): 560,000 EUR                                                                   │
│                                                        

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [26]:
# Cell 17 - Review the final output
print("\n" + "=" * 60)
print("FINAL EXECUTIVE BRIEFING")
print("=" * 60)
print(result.raw)
print()
print("-" * 60)
print(f"Token usage: {result.token_usage}")


FINAL EXECUTIVE BRIEFING
**Financial Exposure Table: Base, Worst, and Best Case Scenarios in EUR**

To calculate the total EUR exposure, we need to consider the following components: rerouting cost delta, SLA penalty clauses triggered, insurance deductible, and opportunity cost of delayed deliveries.

**Base Case:**

* Rerouting cost delta: 12% of the total shipment value, assuming the top choice route (North Sea - Le Havre - Southampton) is implemented. Estimated cost delta: 120,000 EUR (based on 1,000,000 EUR total shipment value)
* SLA penalty clauses triggered: 50,000 EUR (assuming 25% of the affected shipments are subject to SLA penalties)
* Insurance deductible: 20,000 EUR (assuming a standard deductible amount)
* Opportunity cost of delayed deliveries: 100,000 EUR (assuming a 2-day delay and a daily opportunity cost of 50,000 EUR)

Total EUR exposure (base case): 290,000 EUR

**Worst Case:**

* Rerouting cost delta: 22% of the total shipment value, assuming the highest-cost alt

In [27]:
# Cell 18 - Inspect the saved report file
import os

report_file = "globalflow_disruption_report.txt"
if os.path.exists(report_file):
    size = os.path.getsize(report_file)
    print(f"[OK] Report saved: '{report_file}'  ({size} bytes)")
    print()
    with open(report_file, "r") as f:
        print(f.read())
else:
    print("[WARN] Report file not found - check verbose output above.")
    print("Falling back to result.raw:")
    print(result.raw)

[OK] Report saved: 'globalflow_disruption_report.txt'  (2249 bytes)

SITUATION: 
A disruption is affecting several ports and shipping lanes, including port congestion in Rotterdam, a typhoon warning in the South China Sea, and customs delays in Houston. The estimated severity of the disruption is 6/10, with Rotterdam scoring 4/10, the AE-1 shipping lane scoring 8/10 due to the potential typhoon impact, and Houston scoring 3/10. The disruptions are expected to last 3-5 days in Rotterdam, 7-10 days in the South China Sea, and 2-3 days in Houston.

IMPACT: 
The estimated impact on shipments is significant, with 500-700 containers likely to be delayed in Rotterdam, 2,000-3,000 TEUs affected by the typhoon warning in the South China Sea, and 200-300 containers delayed in Houston. The estimated EUR cost exposure is substantial, although the exact figure is not provided. The disruption is expected to cause delays and increased costs, with potential long-term effects on supply chain operations

In [28]:
# Cell 19 - Inspect crew memory (long-term)
import glob

memory_files = glob.glob("*.db") + glob.glob(".crewai/**/*.db", recursive=True)
if memory_files:
    print("Memory database files found:")
    for f in memory_files:
        size = os.path.getsize(f)
        print(f"  {f}  ({size:,} bytes)")
    print()
    print("[TIP] Re-run Cell 16 - agents will reference previous disruption context.")
else:
    print("No memory DB found yet. Run the crew kickoff first (Cell 16).")
    print("Memory DB appears after first successful run.")

No memory DB found yet. Run the crew kickoff first (Cell 16).
Memory DB appears after first successful run.


In [29]:
# Cell 20 - Token usage and rough cost estimate
print("Token Usage Summary")
print("=" * 40)
if hasattr(result, 'token_usage') and result.token_usage:
    tu = result.token_usage
    print(f"  Prompt tokens:     {tu.prompt_tokens:>8,}")
    print(f"  Completion tokens: {tu.completion_tokens:>8,}")
    print(f"  Total tokens:      {tu.total_tokens:>8,}")
    # Groq llama-3.3-70b pricing (approx): $0.59/1M input, $0.79/1M output
    input_cost  = (tu.prompt_tokens     / 1_000_000) * 0.59
    output_cost = (tu.completion_tokens / 1_000_000) * 0.79
    print(f"  Est. cost (70B):   ${input_cost + output_cost:.5f}")
else:
    print("  Token usage not available for this run.")

Token Usage Summary
  Prompt tokens:        6,896
  Completion tokens:    3,595
  Total tokens:        10,491
  Est. cost (70B):   $0.00691


## Extension Tasks

Work through these after the core lab. Estimated time: 30-60 min.

---

### Extension 1 - Add a Financial Analyst Agent

Agent-6 and Task-6 are added and cells are reran.

Create a 6th agent that calculates total EUR exposure from the disruption:

```python
financial_analyst = Agent(
    role="Supply Chain Financial Analyst",
    goal=(
        "Calculate total EUR exposure: rerouting cost delta, "
        "SLA penalty clauses triggered, insurance deductible, "
        "and opportunity cost of delayed deliveries."
    ),
    backstory=(
        "CFA-qualified financial analyst specialising in logistics cost modelling. "
        "Always presents base case, worst case, and best case scenarios."
    ),
    llm=GROQ_SMART,
    verbose=True,
    max_iter=3,
)

task_financial = Task(
    description="Calculate total EUR exposure: rerouting, SLA penalties, insurance.",
    expected_output="Financial exposure table: base / worst / best case in EUR.",
    agent=financial_analyst,
    context=[task_monitor, task_route],
)
```

Add `financial_analyst` to `agents=` and `task_financial` to `tasks=` in the Crew, then re-run.

---

### Extension 2 - Parallel Async Execution

`task_comms` and `task_compliance` are independent and can run in parallel:

```python
task_comms = Task(..., async_execution=True)
task_compliance = Task(..., async_execution=True)
```

Use `crew.kickoff_async()` and compare wall-clock time vs sequential.

---

### Extension 3 - Human-in-the-Loop Gate

Add a human approval step before the report is written:

```python
task_report = Task(..., human_input=True)
```

Re-run - a prompt will appear asking you to approve before the report is written.

---

### Extension 4 - Switch Groq Model Tiers

Try assigning different models per agent based on task complexity:

```python
GROQ_FAST  = "groq/llama-3.1-8b-instant"    # supplier_comms (drafting)
GROQ_SMART = "groq/llama-3.3-70b-versatile" # monitor, router, compliance
```

Compare output quality vs token cost across tiers.